In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env')

placer_api_key = os.getenv('PLACER_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if not placer_api_key:
    raise ValueError("PLACER_API_KEY not found in .env file")
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY not found in .env file")

print("✓ API keys loaded successfully")

## Retail Performance Analysis

**Location:** 1313 Woodbury Ave, Portsmouth, NH
**Coordinates:** latitude 43.0859778, longitude -70.78677

In [ ]:
from pydantic import BaseModel
from typing import Optional, List


class CategoryInfo(BaseModel):
    category: str
    group: str
    subCategory: str


class Address(BaseModel):
    city: str
    state: str
    countryCode: str
    streetName: str
    formattedAddress: str
    shortFormattedAddress: str
    zipCode: str


class RegionDetail(BaseModel):
    code: str
    name: str


class Regions(BaseModel):
    dma: RegionDetail
    state: RegionDetail
    cbsa: RegionDetail


class Venue(BaseModel):
    entityId: str
    entityType: str
    name: str
    categoryInfo: CategoryInfo
    address: Address
    isFlagged: bool
    regions: Regions
    apiId: str
    placerUrl: str
    storeId: Optional[str] = None
    isPermitted: bool


class PlacerResponse(BaseModel):
    data: List[Venue]
    requestId: str


print("✓ Pydantic models defined")

In [ ]:
# Step 1: Search for Placer Entities by specific categories and subcategories
import requests

# Define search criteria using correct Placer taxonomy
# Group -> Category -> SubCategory
search_criteria = [
    {"category": "Breakfast, Coffee, Bakeries & Dessert Shops", "subCategory": None},
    {"category": "Fast Food & QSR", "subCategory": None},
    {"category": "Electronics Stores", "subCategory": None},
    {"category": "Groceries", "subCategory": None},
    {"category": "Home Improvement", "subCategory": None},
    {"category": "Office Supplies", "subCategory": None},
    {"category": "Drugstores & Pharmacies", "subCategory": None},
    {"category": "Superstores", "subCategory": None},
    {"category": "Hotels & Casinos", "subCategory": "Casino"},
    {"category": "Car Wash Services", "subCategory": None}
]

base_url = "https://papi.placer.ai/v1/poi"
headers = {
    "accept": "application/json",
    "x-api-key": f"{placer_api_key}"
}

all_venues = []
venue_ids = set()  # Track unique venues to avoid duplicates

print(f"Searching for venues by category/subcategory within 1 mile radius...\n")

for criteria in search_criteria:
    category = criteria["category"]
    sub_category = criteria["subCategory"]

    # Build query parameters
    params = {
        "lat": 43.0859778,
        "lng": -70.78677,
        "radius": 1,
        "category": category,
        "entityType": "venue",
        "limit": 50,
        "skip": 0
    }

    if sub_category:
        params["subCategory"] = sub_category
        print(f"  Searching: {category} > {sub_category}...", end=" ")
    else:
        print(f"  Searching: {category}...", end=" ")

    try:
        response = requests.get(base_url, params=params, headers=headers)

        if response.status_code == 200:
            result = PlacerResponse.model_validate_json(response.text)

            # Add only unique venues
            new_venues = 0
            for venue in result.data:
                if venue.apiId not in venue_ids:
                    all_venues.append(venue)
                    venue_ids.add(venue.apiId)
                    new_venues += 1

            print(f"Found {new_venues} new venues")
        else:
            print(f"Error: {response.status_code}")

    except Exception as e:
        print(f"Error: {str(e)[:80]}")

venues = all_venues

print(f"\n{'='*100}")
print(f"Total unique venues found: {len(venues)}")
print(f"{'='*100}\n")

# Display summary by category
from collections import defaultdict
category_counts = defaultdict(int)

for venue in venues:
    category_counts[venue.categoryInfo.category] += 1

print("Venues by category:")
for category, count in sorted(category_counts.items()):
    print(f"  {category}: {count}")

print(f"\nAll venues:")
for venue in venues:
    print(f"  - {venue.name} ({venue.categoryInfo.category} > {venue.categoryInfo.subCategory}) - {venue.address.formattedAddress} - {venue.apiId}")

In [ ]:
# Pydantic models for Ranking API response

class RankingDetail(BaseModel):
    rank: int
    percentile: int
    rankedOutOf: int
    regionCode: str


class VenueRanking(BaseModel):
    nationwide: Optional[RankingDetail] = None
    state: Optional[RankingDetail] = None
    dma: Optional[RankingDetail] = None
    cbsa: Optional[RankingDetail] = None
    rankError: Optional[str] = None  # Present when ranking data is missing


class VenueInfo(BaseModel):
    entityId: str
    entityType: str
    name: str
    flagged: bool
    rankedBy: str
    parentChain: Optional[str] = None
    categoryInfo: CategoryInfo


class RankingData(BaseModel):
    visitDurationSegmentation: str
    info: VenueInfo
    apiId: str
    metricType: str
    ranking: VenueRanking


class RankingResponse(BaseModel):
    data: List[RankingData]
    requestId: str


print("✓ Ranking Pydantic models defined (with error handling)")

In [ ]:
# Step 2: Get ranking overview for all venues (batched for API limit of 100)
url = "https://papi.placer.ai/v1/reports/ranking-overview/multi"

# Build apiIds list from venues
api_ids = [venue.apiId for venue in venues]

print(f"Total venues: {len(api_ids)}")
print(f"Fetching rankings in batches of 100...\n")

# Batch requests if we have more than 100 venues
batch_size = 100
all_rankings = []

for i in range(0, len(api_ids), batch_size):
    batch = api_ids[i:i+batch_size]
    batch_num = (i // batch_size) + 1
    total_batches = (len(api_ids) + batch_size - 1) // batch_size

    print(f"Batch {batch_num}/{total_batches}: Fetching {len(batch)} venues...", end=" ")

    payload = {
        "apiIds": batch,
        "period": "last3Months",
        "scope": "category",
        "metric": "visits"
    }

    headers = {
        "accept": "application/json",
        "content-type": "application/json",
        "x-api-key": placer_api_key
    }

    response = requests.post(url, json=payload, headers=headers)

    # Check for API errors
    if response.status_code != 200:
        print(f"\n  Error - Status Code: {response.status_code}")
        print(f"  Response: {response.text}")
        continue

    # Check if response has error structure
    temp_data = response.json()
    if 'code' in temp_data and 'data' not in temp_data:
        print(f"\n  API Error:")
        print(f"    Code: {temp_data.get('code')}")
        print(f"    Message: {temp_data.get('message')}")
        continue

    # Parse response into Pydantic model
    ranking_response = RankingResponse.model_validate_json(response.text)
    all_rankings.extend(ranking_response.data)
    print(f"✓ Got {len(ranking_response.data)} results")

rankings = all_rankings

print(f"\n{'='*100}")
print(f"Total rankings fetched: {len(rankings)}")
print(f"{'='*100}\n")

print(f"Ranking Overview (Last 3 Months - Category Scope):")
print(f"{'Venue':<25} {'Category':<20} {'Nationwide':<12} {'State':<12}")
print("=" * 100)

venues_with_rankings = 0
venues_without_rankings = 0

for ranking in rankings:
    venue_name = ranking.info.name[:24]
    category = ranking.info.categoryInfo.category[:19]

    # Check if ranking data is available (rankError is set when no data)
    if ranking.ranking.rankError:
        nationwide = "No data"
        state = "No data"
        venues_without_rankings += 1
    else:
        nationwide = f"{ranking.ranking.nationwide.percentile}th %ile"
        state = f"{ranking.ranking.state.percentile}th %ile"
        venues_with_rankings += 1

    print(f"{venue_name:<25} {category:<20} {nationwide:<12} {state:<12}")

print("=" * 100)
print(f"\nVenues with ranking data: {venues_with_rankings}")
print(f"Venues without ranking data: {venues_without_rankings}")

In [ ]:
# Pydantic models for Visit Trends API response
from datetime import datetime, timedelta

class VisitTrendsData(BaseModel):
    visitDurationSegmentation: str
    apiId: str
    dates: List[str]
    visits: List[int]
    panelVisits: List[int]


class NoContentError(BaseModel):
    apiId: str
    details: str
    error: str
    code: int


class VisitTrendsResponse(BaseModel):
    data: List[VisitTrendsData | NoContentError]
    requestId: str


# Calculate date range (one year ago to 3 days ago - API needs time to process recent data)
end_date = (datetime.now() - timedelta(days=3)).strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")

print(f"✓ Visit Trends models defined")
print(f"Date range: {start_date} to {end_date}")

In [ ]:
# Step 3: Get visit trends for all venues (day, week, month granularity) - batched
import time

url = "https://papi.placer.ai/v1/reports/visit-trends"

headers = {
    "accept": "application/json",
    "content-type": "application/json",
    "x-api-key": placer_api_key
}

# Build apiIds list from venues
api_ids = [venue.apiId for venue in venues]

print(f"Total venues: {len(api_ids)}")

# Fetch visit trends for day, week, and month granularity
visit_trends_daily = []
visit_trends_weekly = []
visit_trends_monthly = []

batch_size = 100

for granularity, storage in [("day", visit_trends_daily), ("week", visit_trends_weekly), ("month", visit_trends_monthly)]:
    print(f"\nFetching {granularity} visit trends in batches of {batch_size}...")

    # Batch the requests
    for i in range(0, len(api_ids), batch_size):
        batch = api_ids[i:i+batch_size]
        batch_num = (i // batch_size) + 1
        total_batches = (len(api_ids) + batch_size - 1) // batch_size

        print(f"  Batch {batch_num}/{total_batches} ({len(batch)} venues)...", end=" ")

        payload = {
            "apiIds": batch,
            "granularity": granularity,
            "visitDurationSegmentation": "default",
            "startDate": start_date,
            "endDate": end_date
        }

        # Retry for IN_PROGRESS responses
        for attempt in range(10):
            response = requests.post(url, json=payload, headers=headers)

            # Check for API errors (200 and 207 are valid - 207 is Multi-Status for batch operations)
            if response.status_code not in [200, 207]:
                print(f"✗ HTTP {response.status_code}")
                break

            # Check for IN_PROGRESS
            if "IN_PROGRESS" in response.text:
                print(".", end="")
                time.sleep(3)
                continue

            # Try to parse response
            try:
                trends_response = VisitTrendsResponse.model_validate_json(response.text)

                # Separate successful data from NO_CONTENT errors
                batch_count = 0
                for item in trends_response.data:
                    if isinstance(item, VisitTrendsData):
                        storage.append(item)
                        batch_count += 1

                print(f"✓ {batch_count} venues with data")
                break
            except Exception as e:
                print(f"✗ Error: {str(e)[:50]}")
                break

print(f"\n{'='*80}")
print(f"Summary:")
print(f"  Daily data: {len(visit_trends_daily)} venues")
print(f"  Weekly data: {len(visit_trends_weekly)} venues")
print(f"  Monthly data: {len(visit_trends_monthly)} venues")
print(f"{'='*80}")

In [ ]:
# Step 3.5: Calculate distances from original POI using Google Routes API
import googlemaps

# Initialize Google Maps client
gmaps = googlemaps.Client(key=google_api_key)

# Original POI coordinates
origin_lat = 43.0859778
origin_lng = -70.78677

# Calculate distances for each venue using Routes API
venue_distances = {}

print("Calculating distances from original POI (1313 Woodbury Ave)...\n")

for venue in venues:
    address = venue.address.formattedAddress

    try:
        # Use Routes API via the googlemaps library (computeRoutes)
        # First geocode to get destination coordinates
        geocode_result = gmaps.geocode(address)

        if geocode_result:
            venue_location = geocode_result[0]['geometry']['location']
            venue_lat = venue_location['lat']
            venue_lng = venue_location['lng']

            # Use Routes API to compute routes
            url = "https://routes.googleapis.com/directions/v2:computeRoutes"

            payload = {
                "origin": {
                    "location": {
                        "latLng": {
                            "latitude": origin_lat,
                            "longitude": origin_lng
                        }
                    }
                },
                "destination": {
                    "location": {
                        "latLng": {
                            "latitude": venue_lat,
                            "longitude": venue_lng
                        }
                    }
                },
                "travelMode": "DRIVE",
                "routingPreference": "TRAFFIC_AWARE",
                "computeAlternativeRoutes": False,
                "routeModifiers": {
                    "avoidTolls": False,
                    "avoidHighways": False,
                    "avoidFerries": False
                },
                "languageCode": "en-US",
                "units": "IMPERIAL"
            }

            headers = {
                "Content-Type": "application/json",
                "X-Goog-Api-Key": google_api_key,
                "X-Goog-FieldMask": "routes.duration,routes.distanceMeters,routes.polyline.encodedPolyline"
            }

            response = requests.post(url, json=payload, headers=headers)

            if response.status_code == 200:
                result = response.json()

                if 'routes' in result and len(result['routes']) > 0:
                    route = result['routes'][0]
                    distance_meters = route['distanceMeters']
                    distance_miles = distance_meters * 0.000621371
                    duration_seconds = int(route['duration'].rstrip('s'))
                    duration_minutes = duration_seconds // 60

                    venue_distances[venue.apiId] = {
                        'distance_text': f"{distance_miles:.2f} mi",
                        'distance_miles': distance_miles,
                        'duration': f"{duration_minutes} min"
                    }

                    print(f"  {venue.name[:30]:<30} - {distance_miles:.2f} miles ({duration_minutes} min)")
                else:
                    print(f"  {venue.name[:30]:<30} - No route found")
            else:
                print(f"  {venue.name[:30]:<30} - API error: {response.status_code}")
        else:
            print(f"  {venue.name[:30]:<30} - Geocoding failed")

    except Exception as e:
        print(f"  {venue.name[:30]:<30} - Error: {e}")

print(f"\nCalculated distances for {len(venue_distances)} venues")

In [ ]:
# Step 4: Display comprehensive venue performance data in tabular format
print(f"\nVenue Performance Summary ({start_date} to {end_date})")
print(f"{'='*240}")
print(f"{'Venue':<25} {'Category':<20} {'SubCategory':<25} {'Distance':<12} {'Drive Time':<12} {'Nationwide':<12} {'State':<12} {'Daily Avg':<12} {'Weekly Avg':<12} {'Monthly Avg':<13} {'Yearly Total':<15}")
print(f"{'='*240}")

for venue in venues:
    # Find matching data for this venue
    ranking_data = next((r for r in rankings if r.apiId == venue.apiId), None)
    daily_data = next((d for d in visit_trends_daily if d.apiId == venue.apiId), None)
    weekly_data = next((w for w in visit_trends_weekly if w.apiId == venue.apiId), None)
    monthly_data = next((m for m in visit_trends_monthly if m.apiId == venue.apiId), None)
    distance_info = venue_distances.get(venue.apiId)

    # Format venue name, category, and subcategory
    venue_name = venue.name[:24]
    category = venue.categoryInfo.category[:19]
    subcategory = venue.categoryInfo.subCategory[:24]

    # Format distance and duration
    if distance_info:
        distance = distance_info['distance_text']
        duration = distance_info.get('duration', 'N/A')
    else:
        distance = "N/A"
        duration = "N/A"

    # Format ranking data - check for rankError on the ranking object itself
    if ranking_data:
        if ranking_data.ranking.rankError:
            nationwide = "No data"
            state = "No data"
        else:
            nationwide = f"{ranking_data.ranking.nationwide.percentile}th %ile"
            state = f"{ranking_data.ranking.state.percentile}th %ile"
    else:
        nationwide = "N/A"
        state = "N/A"

    # Calculate averages and yearly total
    daily_avg = "N/A"
    weekly_avg = "N/A"
    monthly_avg = "N/A"
    yearly_total = "N/A"

    if daily_data:
        avg_val = sum(daily_data.visits) / len(daily_data.visits) if daily_data.visits else 0
        daily_avg = f"{int(avg_val):,}"
        yearly_total = f"{sum(daily_data.visits):,}"

    if weekly_data:
        avg_val = sum(weekly_data.visits) / len(weekly_data.visits) if weekly_data.visits else 0
        weekly_avg = f"{int(avg_val):,}"
        if yearly_total == "N/A":
            yearly_total = f"{sum(weekly_data.visits):,}"

    if monthly_data:
        avg_val = sum(monthly_data.visits) / len(monthly_data.visits) if monthly_data.visits else 0
        monthly_avg = f"{int(avg_val):,}"
        if yearly_total == "N/A":
            yearly_total = f"{sum(monthly_data.visits):,}"

    print(f"{venue_name:<25} {category:<20} {subcategory:<25} {distance:<12} {duration:<12} {nationwide:<12} {state:<12} {daily_avg:<12} {weekly_avg:<12} {monthly_avg:<13} {yearly_total:<15}")

print(f"{'='*240}")
print(f"\nSummary: {len(visit_trends_daily)} venues with daily data, {len(visit_trends_weekly)} with weekly, {len(visit_trends_monthly)} with monthly")